In [14]:
%pip install openpyxl



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
import numpy as np
import glob
import re
from pathlib import Path


In [16]:
## 2000-2010: https://agreste.agriculture.gouv.fr/agreste-web/disaron/SAANR_DEVELOPPE_2/detail/
## 2010

In [17]:
# -----------------------------
# SETTINGS
# -----------------------------
CROPYIELD_DIR = Path("/Users/marielouiselysholt/Desktop/Master/Masters_2026/EDA/Cropyield")   # folder with FDS_DEVELOPPE_2000.csv ...
LATE_FILE = Path("SAA_2010-2024_definitives_donnees_departementales.xlsx")
LATE_SHEET = "COP"

EARLY_CROP = "Blé tendre d'hiver"
LATE_CROP = "01 - Blé tendre d'hiver et épeautre"

OUTPUT_FILE = "ble_tendre_hiver_yield_2000_2024.csv"

# -----------------------------
# Helper functions
# -----------------------------

def clean_numeric(series):
    s = series.astype(str)
    s = s.str.replace("\u00a0", "", regex=False)
    s = s.str.replace(" ", "", regex=False)
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

def extract_dep_from_libdep(series):
    # "077 - Seine-et-Marne" → "077"
    return series.str.extract(r"^(\d+)")[0].str.zfill(3)


In [18]:
# -----------------------------
# EARLY FILE
# -----------------------------

def clean_numeric(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.replace("\u00a0", "", regex=False).str.strip()
    s = s.str.replace(" ", "", regex=False).str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9.\-]", "", regex=True)
    return pd.to_numeric(s, errors="coerce")

def dep_to_3digits(dep: pd.Series) -> pd.Series:
    """
    Early files have '77', '01', etc + DOM like '971'.
    Convert mainland -> 3 digits (077, 001, ...), keep 3-digit DOM as-is.
    Drops '...' elsewhere.
    """
    d = dep.astype(str).str.strip()
    d = d.replace({"...": np.nan, "": np.nan})
    out = d.copy()

    is_num = out.dropna().str.fullmatch(r"\d+")
    idx = out.dropna().index[is_num]

    for i in idx:
        val = out.loc[i]
        if len(val) <= 3:
            out.loc[i] = val.zfill(3)   # '77' -> '077', '1'->'001', '971'->'971'
        else:
            out.loc[i] = val            # just in case
    return out

# -----------------------------
# EARLY FILES (2000–2010)
# -----------------------------
early_files = sorted(glob.glob(str(CROPYIELD_DIR / "FDS_DEVELOPPE_*.csv")))
early_list = []

for f in early_files:
    year_match = re.search(r"(20\d{2})", Path(f).name)
    if not year_match:
        continue
    year = int(year_match.group(1))
    if year > 2010:
        continue

    # read
    df = pd.read_csv(f, sep=";", encoding="latin1", dtype=str)

    # filter crop
    df = df[df["N306_LIB"].astype(str).str.strip() == EARLY_CROP].copy()
    if df.empty:
        continue

    # remove aggregates like DEP == '...'
    df["DEP"] = df["DEP"].astype(str).str.strip()
    df = df[df["DEP"] != "..."].copy()

    # keep only production + surface (exactly as in your sample)
    df["N027_LIB"] = df["N027_LIB"].astype(str).str.strip()
    df = df[df["N027_LIB"].isin(["Production (volume)", "Superficie développée"])].copy()
    if df.empty:
        continue

    # numeric
    df["VALEUR"] = clean_numeric(df["VALEUR"])

    # classify
    df["_type"] = np.where(df["N027_LIB"].eq("Production (volume)"), "prod", "surf")

    # dep + year
    df["dep"] = dep_to_3digits(df["DEP"])
    df["year"] = pd.to_numeric(df["ANNREF"], errors="coerce")

    df = df.dropna(subset=["dep", "year"])

    # aggregate duplicates then pivot
    g = (
        df.groupby(["year", "dep", "_type"], as_index=False)["VALEUR"]
          .sum()
    )

    wide = (
        g.pivot_table(index=["year", "dep"], columns="_type", values="VALEUR", aggfunc="sum")
         .reset_index()
    )
    wide.columns.name = None

    wide["prod"] = wide.get("prod")
    wide["surf"] = wide.get("surf")
    wide["yield"] = np.where((wide["surf"] > 0) & wide["prod"].notna(), wide["prod"] / wide["surf"], np.nan)
    wide["source"] = "early_csv"

    early_list.append(wide[["year", "dep", "prod", "surf", "yield", "source"]])

early_panel = pd.concat(early_list, ignore_index=True) if early_list else pd.DataFrame(
    columns=["year", "dep", "prod", "surf", "yield", "source"]
)


In [19]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Your late file + sheet
LATE_FILE  = CROPYIELD_DIR / "SAA_2010-2024_définitives_donnees_departementales.xlsx"
LATE_SHEET = "COP"

# IMPORTANT: in the late Excel, winter wheat is labeled like this
LATE_CROP = "01 - Blé tendre d'hiver et épeautre"

def clean_numeric(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.replace("\u00a0", "", regex=False).str.strip()
    s = s.str.replace(" ", "", regex=False).str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9.\-]", "", regex=True)
    return pd.to_numeric(s, errors="coerce")

def extract_dep_from_libdep(series: pd.Series) -> pd.Series:
    # "077 - Seine-et-Marne" -> "077"
    return series.astype(str).str.extract(r"^(\d+)")[0].str.zfill(3)

def find_header_row(excel_path: Path, sheet: str, needle: str = "LIB_SAA", max_scan: int = 80) -> int:
    raw = pd.read_excel(excel_path, sheet_name=sheet, header=None, dtype=str)
    for i in range(min(max_scan, len(raw))):
        if raw.iloc[i].astype(str).str.contains(needle, na=False).any():
            return i
    raise ValueError(f"Could not find header row containing '{needle}' in sheet '{sheet}'.")

# -----------------------------
# LATE FILE (2010–2024)
# -----------------------------

# COP sheet has a few metadata rows above the header, so detect the true header row
header_row = find_header_row(LATE_FILE, LATE_SHEET, needle="LIB_SAA")

late = pd.read_excel(LATE_FILE, sheet_name=LATE_SHEET, header=header_row, dtype=str)

late = late[late["LIB_SAA"].astype(str).str.strip() == LATE_CROP].copy()

late["reg"] = extract_dep_from_libdep(late["LIB_REG2"])

# Use only years that actually exist as both SURF_YYYY and PROD_YYYY
surf_years = {int(c.split("_")[1]) for c in late.columns if re.fullmatch(r"SURF_\d{4}", str(c))}
prod_years = {int(c.split("_")[1]) for c in late.columns if re.fullmatch(r"PROD_\d{4}", str(c))}
years = sorted(surf_years & prod_years)

late_rows = []
for y in years:
    surf_col = f"SURF_{y}"
    prod_col = f"PROD_{y}"

    temp = late[["dep", surf_col, prod_col]].copy()
    temp["year"] = y
    temp["surf"] = clean_numeric(temp[surf_col])
    temp["prod"] = clean_numeric(temp[prod_col])
    temp["yield"] = np.where((temp["surf"] > 0) & temp["prod"].notna(), temp["prod"] / temp["surf"], np.nan)
    temp["source"] = "late_excel"

    late_rows.append(temp[["year", "dep", "prod", "surf", "yield", "source"]])

late_panel = pd.concat(late_rows, ignore_index=True) if late_rows else pd.DataFrame(
    columns=["year", "dep", "prod", "surf", "yield", "source"]
)


In [20]:
# -----------------------------
# MERGE PANELS
# Prefer late from 2011 onward
# -----------------------------

panel = pd.concat([
    early_panel[early_panel["year"] < 2011],
    late_panel[late_panel["year"] >= 2011]
])

panel = panel.sort_values(["year", "reg"]).reset_index(drop=True)

panel.to_csv(OUTPUT_FILE, index=False)

print("Panel created:", OUTPUT_FILE)
print("Years:", panel.year.min(), "-", panel.year.max())
print("Regions:", panel.reg.nunique())

Panel created: ble_tendre_hiver_yield_2000_2024.csv
Years: 2011 - 2024
Departments: 98


/var/folders/g6/v5w89nm96b732r3v5w91v8dm0000gn/T/ipykernel_66343/3057164157.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  panel = pd.concat([
